[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/26-vision-language-models.ipynb)

# Module 26: Vision-Language Models

Companion notebook to [Module 26](https://transformers-and-math.drmhse.com/modules/26-vision-language-models) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — Three connectors (section 26.1)

In [ ]:
d_vis, d_text = 1024, 2048
patches = torch.randn(1, 576, d_vis)                     # a 24 × 24 grid from the vision encoder

mlp = nn.Sequential(nn.Linear(d_vis, d_text), nn.GELU(), nn.Linear(d_text, d_text))
print('MLP projection:   ', tuple(mlp(patches).shape))

queries = nn.Parameter(torch.randn(1, 64, d_text))
cross = nn.MultiheadAttention(d_text, 8, kdim=d_vis, vdim=d_vis, batch_first=True)
print('resampler:        ', tuple(cross(queries, patches, patches)[0].shape))

grid = patches.view(1, 24, 24, d_vis)
merged = grid.view(1, 12, 2, 12, 2, d_vis).permute(0, 1, 3, 2, 4, 5).reshape(1, 144, 4 * d_vis)
print('2×2 patch merging:', tuple(nn.Linear(4 * d_vis, d_text)(merged).shape))

## Part 2 — Tiling a large image (section 26.2)

In [ ]:
def tiles(w, h, tile=336, per_tile=576):
    cols, rows = math.ceil(w / tile), math.ceil(h / tile)
    n = cols * rows + 1                                   # plus the overview tile
    return cols, rows, n * per_tile

for w, h in [(1008, 336), (672, 672), (1344, 1344)]:
    c, r, t = tiles(w, h)
    print(f'{w}×{h}: {c}×{r} grid, {t:,} image tokens, {t // 4:,} after 2×2 merging')

## Part 3 — One training example and its loss mask (section 26.4)

In [ ]:
vocab, d = 1000, 64
prefix = torch.randint(0, vocab, (8,))                    # "User: What is this?"
answer = torch.randint(0, vocab, (10,))                   # "Assistant: It's a red fox."
image_vectors = torch.randn(576, d)                       # from the connector
embed, head = nn.Embedding(vocab, d), nn.Linear(d, vocab)

inputs = torch.cat([embed(prefix), image_vectors, embed(answer)])          # one sequence of 594 vectors
targets = torch.cat([torch.full((8 + 576,), -100), answer])                # only the answer is scored
targets = torch.cat([targets[1:], torch.tensor([-100])])                   # position t predicts token t + 1
logits = head(inputs)                                                       # (a real model runs the transformer first)
loss = F.cross_entropy(logits, targets, ignore_index=-100)
print(f'{len(inputs)} positions, {(targets != -100).sum().item()} scored; loss {loss.item():.3f}')

## Your turn
1. How many image tokens does a 2016 × 1008 image cost with this tiling, before and after merging?
2. Remove the `-100` mask so every position is scored. Why is predicting "image tokens" meaningless?